# Tracking de experimentos y diagnóstico de entrenamiento

Tres flujos: (a) un run de entrenamiento con auditoría de gradientes por canal,
(b) un study de hiperparámetros con objetivo compuesto y poda, y (c) seguir o
reanudar un study desde disco. Todo queda en `.soma/` para que un front lo lea.


In [1]:
import json, pathlib, time
import torch
import torch.nn as nn

import soma
from soma import ChannelConfig, DifferentiableFilter, Graph, search

# Aviso benigno de PyTorch sobre backward hooks (el audit los usa a
# propósito); la suite de tests lo silencia igual.
import warnings
warnings.filterwarnings("ignore", message="Full backward hook is firing")


## (a) Run de entrenamiento con auditoría

`track_run` crea `.soma/runs/<run_id>/` (manifest con git/host, status con
heartbeat, topología del grafo, events/metrics.jsonl). `gradient_audit` con
`channels=` añade diagnósticos por canal: canales muertos, dormidos (Sokar
2023), ignorados (gradient starvation) y leakage entre grupos (CKA).


In [2]:
class Encoder(DifferentiableFilter):
    _cache_version = "nb06-encoder-v1"
    lr: float = search(1e-3, 1e-1, scale="log")

    def build_module(self, input_shape):
        return nn.Sequential(nn.Linear(input_shape[-1], 16), nn.ReLU(), nn.Linear(16, 8))

    def output_shape(self, input_shape):
        return (*input_shape[:-1], 8)

g = Graph()
g.node("encoder", Encoder())
x = torch.randn(64, 12)
y = torch.randn(64, 8)
g.materialize(x)
g.train()
g.make_optimizer(lr=0.01)


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)

In [3]:
with g.track_run("baseline", tags=["demo"]) as run:
    cfg = ChannelConfig(snapshot_every=10,
                        groups={"encoder": {"a": range(0, 4), "b": range(4, 8)}})
    with g.gradient_audit(channels=cfg) as audit:
        module = dict(g.filters())["encoder"]._module
        for epoch in range(5):
            run.log_epoch(epoch, total=5)
            with g.context() as ctx:
                g.zero_grad()
                loss = ((module(x) - y) ** 2).mean()
                g.backward(ctx, loss)   # snapshot del audit + StepCompleted
            g.step(ctx)
            run.log("loss", loss.detach().item(), step=epoch)
    print(audit.report().pretty())

run_dir = pathlib.Path(run.dir)
print(sorted(p.name for p in run_dir.iterdir()))


                  filter   steps      act|μ|       act σ      |out∂|        |θ∂|         |θ|         ∂/θ                     flags
----------------------------------------------------------------------------------------------------------------------------------
                 encoder       5   1.804e-01   2.139e-01   8.869e-02   2.276e-01   2.907e+00   7.826e-02                   HEALTHY
['diagnostics', 'events.jsonl', 'graph.json', 'graph.mmd', 'manifest.json', 'metrics.jsonl', 'status.json']


In [4]:
# Todo lo que un front necesita: eventos, métricas y diagnósticos
print((run_dir / "graph.mmd").read_text())
print(json.loads((run_dir / "status.json").read_text()))
print((run_dir / "diagnostics" / "report.json").read_text()[:400])


graph LR
    encoder[encoder]

{'state': 'completed', 'updated_at': '2026-07-30T08:24:57.002101406Z', 'heartbeat_at': '2026-07-30T08:24:57.002101406Z', 'finished_at': '2026-07-30T08:24:57.002101406Z'}
{
  "n_steps": 5,
  "filters": [
    {
      "filter": "encoder",
      "n_steps": 5,
      "metrics": {
        "act_mean_abs": 0.1804320454597473,
        "act_std": 0.21390874981880187,
        "act_zero_frac": 0.0,
        "act_zero_frac_max": 0.0,
        "act_sat_frac_max": 0.0,
        "out_grad_norm": 0.08869450241327285,
        "out_grad_max": 0.01318521611392498,
        "param_grad_nor


## (b) Study con grid, objetivo compuesto y poda

El espacio sale de los descriptores `search()` de los filtros
(`graph.search_space()`, nombres `nodo.param`). El objetivo puede ser un
callable sobre las métricas; la poda usa la regla de la mediana vía
`trial.report()`.


In [5]:
study = g.study("demo-grid", strategy="grid", n_trials=3,
                objective=lambda m: m["fit"] - 0.1 * m["cost"],
                direction="maximize", pruning=("median", 2))

def train(trial):
    g.apply_params(trial.params)
    lr = trial["encoder.lr"]
    for step in range(8):
        fit = 1.0 - abs(lr - 0.01) * 10 + step * 0.01
        if trial.report("fit", fit, step):
            return None                      # podado
    return {"fit": fit, "cost": lr * 100}

# Eventos en vivo: solo el progreso agregado (TrialMetric llega uno por
# report(); imprimirlos todos es ruido).
def progress(e):
    if e["event_type"] == "StudyProgress":
        print(f" → {e['completed']}/{e['total']} trials, best={e['best_value']:.3f}")

study.run(train, on_event=progress)
time.sleep(0.3)   # el callback de eventos es asíncrono: deja drenar las últimas líneas
for t in study.trials:
    mark = "✂" if t["state"] == "pruned" else "✓"
    print(f" {mark} {t['id']}  {t['state']}")
best = study.best_trial
print(f"best: {best['id']}  params={best['params']}  score={best['metrics']['score']:.3f}")
print("run dir:", study.run_dir)


 → 1/3 trials, best=0.970
 → 2/3 trials, best=0.970
 → 3/3 trials, best=0.970


 ✓ trial_0000  completed
 ✓ trial_0001  completed
 ✓ trial_0002  completed
best: trial_0000  params={'encoder.lr': 0.0010000000000000002}  score=0.970
run dir: .soma/runs/study_20260730T082457_6300


## (c) Seguir y reanudar desde disco

`study.json` se reescribe atómicamente tras cada trial: desde cualquier
máquina con acceso al directorio se puede cargar el estado, y `resume=True`
continúa exactamente donde quedó (sin repetir puntos del grid).


In [6]:
reloaded = soma.Study.load(study.run_dir)
print(reloaded.progress, len(reloaded.trials))
# reloaded.run(train, resume=True)   # continuaría si quedaran trials

for exp in soma.experiments():
    print(exp["name"], exp["metrics"], exp["tags"])


1.0 3
baseline {'loss': 0.9568052291870117} ['demo', 'run:run_20260730T082456_4d27']
demo-grid {'fit': 0.98, 'cost': 0.10000000000000002, 'score': 0.97} ['run:study_20260730T082457_6300']
